# PDF Chunking — Built-in AI Functions

Second half of the split built-in pipeline: reads the structured parses cached by `PDF Parsing (built-in AI functions)` in `rag_chatbot_dev.bronze.doc_parsed_builtin`, splits them into semantic chunks with `ai_prep_search()`, and writes the chunks to `rag_chatbot_dev.silver.doc_chunks_builtin` for downstream RAG use and for side-by-side experimentation with the pypdf pipeline (`doc_chunks_pypdf`). Parsing is decoupled, so re-running this notebook involves no model-backed parsing cost.

In [0]:
# --- Configuration -------------------------------------------------------
CATALOG = "rag_chatbot_dev"
PARSE_TABLE = f"{CATALOG}.bronze.doc_parsed_builtin"   # parse cache written by PDF Parsing (built-in AI functions)
CHUNK_TABLE = f"{CATALOG}.silver.doc_chunks_builtin"   # target Delta table (silver layer)

# Parsing is decoupled: read the cached parse output instead of re-running
# the model-backed ai_parse_document on every chunking run.
parsed_df = spark.table(PARSE_TABLE)
parsed_df.createOrReplaceTempView("parsed_docs")

print(f"Using {parsed_df.count()} cached parse(s) from {PARSE_TABLE}")
display(parsed_df.select("doc_path", "parsed_at"))

Using 4 cached parse(s) from rag_chatbot_dev.bronze.doc_parsed_builtin


doc_path,parsed_at
/Volumes/rag_chatbot_dev/bronze/docs/expense_policy.pdf,2026-11-07T16:20:26.534Z
/Volumes/rag_chatbot_dev/bronze/docs/hr_leave_policy.pdf,2026-11-07T16:20:26.534Z
/Volumes/rag_chatbot_dev/bronze/docs/incident_runbook.pdf,2026-11-07T16:20:26.534Z
/Volumes/rag_chatbot_dev/bronze/docs/product_faq.pdf,2026-11-07T16:20:26.534Z


In [0]:
# Chunk the parsed documents with ai_prep_search (semantic chunking):
#   content          -> chunk_to_retrieve: raw chunk text
#   content_to_embed -> chunk_to_embed: chunk text enriched with title / section headers
prepped_df = spark.sql("""
    SELECT doc_path, ai_prep_search(parsed) AS prepped
    FROM parsed_docs
""")
prepped_df.createOrReplaceTempView("prepped_docs")

# Flatten into one row per chunk, with the same table shape as doc_chunks_pypdf
spark.sql("""
    SELECT
        md5(concat_ws('_', doc_path, chunk.value:chunk_position::STRING)) AS chunk_id,
        doc_path,
        element_at(split(doc_path, '/'), size(split(doc_path, '/'))) AS doc_name,
        chunk.value:chunk_position::INT     AS chunk_num,
        chunk.value:chunk_to_retrieve::STRING AS content,
        chunk.value:chunk_to_embed::STRING    AS content_to_embed,
        length(chunk.value:chunk_to_retrieve::STRING) AS num_chars,
        current_timestamp() AS ingested_at
    FROM prepped_docs,
    LATERAL variant_explode(prepped:document:contents) AS chunk
""").createOrReplaceTempView("chunks_final")

num_chunks = spark.table("chunks_final").count()
num_docs = spark.table("chunks_final").select("doc_path").distinct().count()
print(f"Produced {num_chunks} chunk(s) from {num_docs} document(s)")
display(spark.table("chunks_final").select("doc_name", "chunk_num", "num_chars"))

Produced 4 chunk(s) from 4 document(s)


doc_name,chunk_num,num_chars
expense_policy.pdf,0,900
hr_leave_policy.pdf,0,913
incident_runbook.pdf,0,1000
product_faq.pdf,0,879


In [0]:
from pyspark.sql import functions as F

# Target table: primary key + change data feed so it is ready to back a
# Vector Search Delta Sync index later (mirrors doc_chunks_pypdf).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CHUNK_TABLE} (
    chunk_id         STRING NOT NULL,
    doc_path         STRING NOT NULL,
    doc_name         STRING,
    chunk_num        INT,
    content          STRING,
    content_to_embed STRING,
    num_chars        INT,
    ingested_at      TIMESTAMP,
    CONSTRAINT doc_chunks_builtin_pk PRIMARY KEY (chunk_id)
) TBLPROPERTIES (delta.enableChangeDataFeed = true);
""")

# Full refresh per document: delete every row of each ingested PDF, then
# insert its new chunks, so re-runs never leave stale or duplicate chunks
# behind when AI chunk boundaries shift between runs. The doc names are
# inlined as literals because the view contains current_timestamp(), which
# Delta rejects as a non-deterministic DELETE predicate; matching on the
# stable file name also makes the refresh robust to path format changes.
doc_names = [
    row.doc_name
    for row in spark.table("chunks_final").select("doc_name").distinct().collect()
]
if doc_names:
    names_sql = ", ".join(f"'{n.replace(chr(39), chr(39) * 2)}'" for n in doc_names)
    # Replace all chunks of every document currently in the volume
    spark.sql(f"DELETE FROM {CHUNK_TABLE} WHERE doc_name IN ({names_sql})")
    # Purge chunks of documents that no longer exist in the volume, so no
    # old data remains after a file is removed
    spark.sql(f"DELETE FROM {CHUNK_TABLE} WHERE doc_name NOT IN ({names_sql})")
else:
    # Volume has no PDFs left: clear the table so it stays consistent
    spark.sql(f"DELETE FROM {CHUNK_TABLE} WHERE TRUE")

spark.sql(f"""
INSERT INTO {CHUNK_TABLE}
SELECT * FROM chunks_final
""")

# --- Verification --------------------------------------------------------
print(f"Total rows in {CHUNK_TABLE}: {spark.table(CHUNK_TABLE).count()}")
display(
    spark.table(CHUNK_TABLE)
    .groupBy("doc_name")
    .agg(
        F.count("*").alias("chunks"),
        F.min("chunk_num").alias("first_chunk"),
        F.max("chunk_num").alias("last_chunk"),
        F.sum("num_chars").alias("total_chars"),
    )
)
display(spark.table(CHUNK_TABLE).orderBy("doc_name", "chunk_num").limit(5))

Total rows in rag_chatbot_dev.silver.doc_chunks_builtin: 4


doc_name,chunks,first_chunk,last_chunk,total_chars
expense_policy.pdf,1,0,0,900
hr_leave_policy.pdf,1,0,0,913
incident_runbook.pdf,1,0,0,1000
product_faq.pdf,1,0,0,879


chunk_id,doc_path,doc_name,chunk_num,content,content_to_embed,num_chars,ingested_at
c3a3de8911ae0699404e47a3b4170727,/Volumes/rag_chatbot_dev/bronze/docs/expense_policy.pdf,expense_policy.pdf,0,"Scope: This policy applies to all employees who submit business expenses. Travel Booking: Flights must be booked at least 14 days before departure when possible. Economy class is the standard for flights under 6 hours. Business class is allowed for flights over 6 hours with VP approval. Hotels: The nightly limit is 180 dollars in most cities and 250 dollars in London, New York, and Tokyo. Breakfast is reimbursed up to 20 dollars per day. Meals during travel: Up to 60 dollars per day, with receipts required for any single meal above 25 dollars. Submission: Expense reports must be submitted within 30 days of the expense date through the Finance portal. Late submissions over 90 days are not reimbursed. Approval: Expenses under 500 dollars are approved by the direct manager. Expenses between 500 and 5000 dollars require the department head. Expenses above 5000 dollars require the CFO.","Company: ACME CORP Currency: dollars Policy Title: ACME CORP EXPENSE POLICY Section: ACME CORP EXPENSE POLICY Created by ACME CORP, this is an expense policy document. Content: Scope: This policy applies to all employees who submit business expenses. Travel Booking: Flights must be booked at least 14 days before departure when possible. Economy class is the standard for flights under 6 hours. Business class is allowed for flights over 6 hours with VP approval. Hotels: The nightly limit is 180 dollars in most cities and 250 dollars in London, New York, and Tokyo. Breakfast is reimbursed up to 20 dollars per day. Meals during travel: Up to 60 dollars per day, with receipts required for any single meal above 25 dollars. Submission: Expense reports must be submitted within 30 days of the expense date through the Finance portal. Late submissions over 90 days are not reimbursed. Approval: Expenses under 500 dollars are approved by the direct manager. Expenses between 500 and 5000 dollars require the department head. Expenses above 5000 dollars require the CFO.",900,2026-11-08T14:08:41.060Z
4f1c996a1183318a3543fc62fc8ff4df,/Volumes/rag_chatbot_dev/bronze/docs/hr_leave_policy.pdf,hr_leave_policy.pdf,0,All full-time employees receive 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per month. Unused leave up to 5 days may be carried over to the next year. Any balance beyond 5 days expires on March 31. Employees receive 12 days of paid sick leave per year. A doctor's note is required for sick leave longer than 3 consecutive days. Sick leave does not carry over. Primary caregivers receive 26 weeks of fully paid parental leave. Secondary caregivers receive 8 weeks of fully paid leave. Leave must begin within 12 months of the child's arrival. Submit requests through the People Portal at least 10 working days in advance for leave longer than 5 days. Short leave of 1 to 2 days requires manager approval only. Requests are answered within 2 working days. Acme Corp observes 10 public holidays per year. The holiday calendar is published in the People Portal each December.,Company: Acme Corp Document Type: Leave Policy Title: Acme Corp Leave Policy (Version 3.2) Version: 3.2 Document Title: Acme Corp Leave Policy (Version 3.2) Sections: Annual Leave; Sick Leave; Parental Leave; How to Request Leave; Public Holidays Acme Corp created a Leave Policy document. Content: All full-time employees receive 24 days of paid annual leave per calendar year. Leave accrues monthly at 2 days per month. Unused leave up to 5 days may be carried over to the next year. Any balance beyond 5 days expires on March 31. Employees receive 12 days of paid sick leave per year. A doctor's note is required for sick leave longer than 3 consecutive days. Sick leave does not carry over. Primary caregivers receive 26 weeks of fully paid parental leave. Secondary caregivers receiv